# Telco Churn: synthetic smoke-эксперимент

Исполненный автономный эксперимент на **детерминированных синтетических smoke-данных**.
Он проверяет реальный код репозитория, но не оценивает качество на исходном публичном наборе.

## tl;dr

Ниже показан фактически исполненный smoke-run: объём синтетики, выбранная на validation
модель и метрики неизменяемого synthetic test split. Эти числа нельзя переносить на реальные данные.

In [1]:
import io
import json
import tempfile
from contextlib import redirect_stdout
from pathlib import Path

import pandas as pd
from IPython.display import display

from telco_churn.data import TARGET, split_data, validate_frame
from telco_churn.evaluate import main as evaluate_model
from telco_churn.generate_smoke_data import generate_smoke_frame
from telco_churn.train import main as train_model

SEED = 20250809
ROWS = 400
smoke_frame = generate_smoke_frame(rows=ROWS, seed=SEED)
validated_frame = validate_frame(smoke_frame)
train_frame, validation_frame, test_frame = split_data(validated_frame, seed=SEED)

temporary_directory = tempfile.TemporaryDirectory(prefix="telco-smoke-")
run_directory = Path(temporary_directory.name)
data_path = run_directory / "smoke.csv"
artifact_path = run_directory / "model.joblib"
validation_path = run_directory / "validation.json"
metrics_path = run_directory / "test_metrics.json"
errors_path = run_directory / "test_errors.csv"
importance_path = run_directory / "permutation_importance.csv"
validated_frame.to_csv(data_path, index=False)

with redirect_stdout(io.StringIO()):
    train_model(
        [
            "--data",
            str(data_path),
            "--artifact",
            str(artifact_path),
            "--report",
            str(validation_path),
            "--seed",
            str(SEED),
        ]
    )
    evaluate_model(
        [
            "--data",
            str(data_path),
            "--artifact",
            str(artifact_path),
            "--metrics",
            str(metrics_path),
            "--errors",
            str(errors_path),
            "--importance",
            str(importance_path),
        ]
    )

validation_report = json.loads(validation_path.read_text(encoding="utf-8"))
test_metrics = json.loads(metrics_path.read_text(encoding="utf-8"))
error_rows = pd.read_csv(errors_path)
importance_rows = pd.read_csv(importance_path)
summary = pd.DataFrame(
    [
        {
            "data": "deterministic synthetic smoke",
            "rows": len(validated_frame),
            "churn_rate": validated_frame[TARGET].mean(),
            "selected_model": test_metrics["model_name"],
            "test_pr_auc": test_metrics["pr_auc"],
            "test_roc_auc": test_metrics["roc_auc"],
            "test_f1": test_metrics["f1"],
            "top_permutation_feature": importance_rows.iloc[0]["feature"],
        }
    ]
)
summary.round(4)

,data,rows,churn_rate,selected_model,test_pr_auc,test_roc_auc,test_f1,top_permutation_feature
0,deterministic synthetic smoke,400,0.1775,random_forest,0.3188,0.6872,0.3333,Contract


## Context & Methods


Цель — проверить полный churn workflow на локальной синтетике. Настоящие `train.main` и
`evaluate.main` сравнивают Dummy и основные модели, выбирают pipeline/порог на validation,
затем считают synthetic test-метрики, сегментные ошибки и permutation importance.


            ### Key Assumptions


- Seed `20250809`, 400 синтетических клиентов; `customerID` не входит в признаки.
- Пустой `TotalCharges` для нулевого tenure обрабатывается imputer внутри pipeline.
- Бюджет удержания 20% — учебное предположение, не подтверждённый бизнес-лимит.
- Permutation importance описывает предсказания smoke-модели, а не причинность оттока.

## Data

Smoke-таблица создаётся локальным генератором с фиксированным seed, затем проходит ту же
проверку схемы и то же разбиение, что и пользовательский CSV. Сетевые источники не используются.

In [2]:
display(
    pd.DataFrame(
        {
            "split": ["train", "validation", "test"],
            "rows": [len(train_frame), len(validation_frame), len(test_frame)],
            "churn_rate": [
                train_frame[TARGET].mean(),
                validation_frame[TARGET].mean(),
                test_frame[TARGET].mean(),
            ],
            "missing_total_charges": [
                train_frame["TotalCharges"].isna().sum(),
                validation_frame["TotalCharges"].isna().sum(),
                test_frame["TotalCharges"].isna().sum(),
            ],
        }
    ).round(4)
)
display(validated_frame.head(3))

,split,rows,churn_rate,missing_total_charges
0,train,240,0.1792,1
1,validation,80,0.1750,2
2,test,80,0.1750,1


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,SMOKE-00000,Male,0,Yes,No,0,Yes,No phone service,No,No internet service,...,No internet service,No internet service,No internet service,No internet service,One year,Yes,Credit card (automatic),26.07,NaN,0
1,SMOKE-00001,Female,0,Yes,No,54,Yes,Yes,DSL,No,...,Yes,No,Yes,No,Month-to-month,No,Bank transfer (automatic),73.15,3799.34,0
2,SMOKE-00002,Female,0,Yes,No,57,Yes,Yes,No,No internet service,...,No internet service,No internet service,No internet service,No internet service,Month-to-month,Yes,Bank transfer (automatic),61.74,3542.11,0


## Results

Первая таблица — сравнение кандидатов на validation. Следующие результаты относятся только
к synthetic test split; таблица ошибок ограничена несколькими строками.

In [3]:
validation_table = pd.DataFrame(validation_report["models"]).T
display(
    validation_table[["pr_auc", "roc_auc", "precision", "recall", "f1", "selected_fraction"]]
    .sort_values("pr_auc", ascending=False)
    .round(4)
)
display(
    pd.DataFrame(
        {
            "metric": ["pr_auc", "roc_auc", "precision", "recall", "f1"],
            "synthetic_test": [
                test_metrics[key] for key in ["pr_auc", "roc_auc", "precision", "recall", "f1"]
            ],
        }
    ).round(4)
)
display(importance_rows.head(8).round(4))

,pr_auc,roc_auc,precision,recall,f1,selected_fraction
random_forest,0.575424,0.850649,0.5,0.571429,0.533333,0.2
weighted_logistic_regression,0.476436,0.807359,0.375,0.428571,0.4,0.2
decision_tree,0.392236,0.725649,0.375,0.428571,0.4,0.2
gradient_boosting,0.354945,0.720779,0.3125,0.357143,0.333333,0.2
dummy,0.175,0.5,0.3125,0.357143,0.333333,0.2


,metric,synthetic_test
0,pr_auc,0.3188
1,roc_auc,0.6872
2,precision,0.3125
3,recall,0.3571
4,f1,0.3333


,feature,importance_mean,importance_std
0,Contract,0.0767,0.0085
1,PaymentMethod,0.0356,0.0108
2,StreamingMovies,0.0325,0.0131
3,OnlineSecurity,0.0151,0.0275
4,MultipleLines,0.0146,0.0071
5,StreamingTV,0.0132,0.0150
6,PhoneService,0.0100,0.0121
7,tenure,0.0084,0.0649


In [4]:
error_rows.head(8)

,customerID,Contract,InternetService,tenure,Churn,score,prediction,error_type
0,SMOKE-00022,Month-to-month,Fiber optic,18,0,0.726669,1,false_positive
1,SMOKE-00167,Month-to-month,DSL,33,0,0.706371,1,false_positive
2,SMOKE-00319,Month-to-month,DSL,18,0,0.653961,1,false_positive
3,SMOKE-00078,Month-to-month,Fiber optic,11,0,0.610114,1,false_positive
4,SMOKE-00164,Month-to-month,DSL,17,0,0.602815,1,false_positive
5,SMOKE-00221,Month-to-month,DSL,22,0,0.596572,1,false_positive
6,SMOKE-00151,Month-to-month,Fiber optic,20,0,0.580569,1,false_positive
7,SMOKE-00328,Month-to-month,DSL,27,0,0.575505,1,false_positive


## Takeaways

Выводы ниже сформированы из сохранённых outputs текущего запуска и относятся только к smoke-проверке.

In [5]:
print(
    f"- На synthetic test выбран {test_metrics['model_name']}: "
    f"PR-AUC={test_metrics['pr_auc']:.4f}, ROC-AUC={test_metrics['roc_auc']:.4f}, "
    f"F1={test_metrics['f1']:.4f}."
)
print(
    f"- Верхний synthetic permutation feature: {importance_rows.iloc[0]['feature']}; "
    f"строк FP/FN: {len(error_rows)}."
)
print("- Эти outputs подтверждают только исполнимость churn pipeline, не качество на IBM Telco.")

- На synthetic test выбран random_forest: PR-AUC=0.3188, ROC-AUC=0.6872, F1=0.3333.
- Верхний synthetic permutation feature: Contract; строк FP/FN: 20.
- Эти outputs подтверждают только исполнимость churn pipeline, не качество на IBM Telco.
